In [42]:
# Import necessary libraries and dependencies
import pandas as pd
from datetime import datetime, timedelta

In [43]:
# Load the dataset to a pandas DataFrame
zulo_bank = pd.read_csv('dataset/zulo_bank.csv')

In [44]:
# Display the first 5 rows of the dataset
# display(zulo_bank.head(5))

In [45]:
# Check the data set
# zulo_bank.info()

In [46]:
# Fill up the missing values with the appropriate parameters
zulo_bank.fillna(
    {
        "LoanAmount": 0.0,
        "LoanType": "Unknown",
        "InterestRate": 0.0,
    },
    inplace=True,
)

,FullName,Email,Phone,TransactionType,Amount,TransactionDate,AccountType,Balance,OpeningDate,LoanAmount,LoanType,StartDate,EndDate,InterestRate
0,Carol Miller,yfisher@example.org,6088279027,withdrawal,102.15,2023-04-26,Savings,5652.16,2019-08-12,0.00,Unknown,NaN,NaN,0.00
1,Geoffrey Banks,gonzalesgeorge@example.net,001-546-857-6518x5359,withdrawal,358.80,2020-06-13,Credit,2881.24,2019-05-06,32428.90,Mortgage,2021-06-24,2050-01-08 04:59:17.907588,2.12
2,Geoffrey Banks,gonzalesgeorge@example.net,001-546-857-6518x5359,withdrawal,358.80,2020-06-13,Credit,2881.24,2019-05-06,31406.77,Personal,2021-02-27,2038-10-12 04:59:17.907821,4.63
3,Geoffrey Banks,gonzalesgeorge@example.net,001-546-857-6518x5359,withdrawal,358.80,2020-06-13,Credit,2881.24,2019-05-06,27834.00,Personal,2019-12-05,2037-08-15 04:59:17.909497,2.15
4,Geoffrey Banks,gonzalesgeorge@example.net,001-546-857-6518x5359,withdrawal,358.80,2020-06-13,Credit,2881.24,2019-05-06,27873.08,Auto,2022-01-19,2037-06-03 04:59:17.913974,7.03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1549,Norman Lee,keithgarcia@example.org,+1-225-907-5033,deposit,484.00,2020-12-27,Checking,6463.29,2023-10-07,14085.72,Personal,2019-04-24,2038-11-17 04:59:17.905253,2.96
1550,Norman Lee,keithgarcia@example.org,+1-225-907-5033,deposit,484.00,2020-12-27,Checking,6463.29,2023-10-07,26011.89,Personal,2020-01-11,2049-05-18 04:59:17.914544,6.82
1551,Eric Anderson,spencer16@example.com,851-945-5888,withdrawal,427.36,2023-06-19,Savings,3308.76,2019-09-02,0.00,Unknown,NaN,NaN,0.00
1552,Ryan Alexander,loganjohn@example.org,900.377.1792x148,withdrawal,415.01,2022-02-09,Checking,6273.50,2019-07-19,18162.60,Personal,2023-11-09,2028-11-08 04:59:17.906084,2.12


In [47]:
# Check the data set again after filling missing values
# zulo_bank.info()

In [48]:
# Modeling for the database:

"""
Convert to 1NF: First Normal Form
Splitting the 'FullName' column into 'first_name' and 'last_name'
"""
zulo_bank[["first_name", "last_name"]] = zulo_bank["FullName"].str.split(expand=True)

In [49]:
"""
Converting 1NF to 2NF: Second Normal Form
Customers table
"""
customer = (
    zulo_bank[["first_name", "last_name", "Email", "Phone"]]
    .copy()
    .drop_duplicates()
    .reset_index(drop=True)
)
customer["customer_id"] = range(1, len(customer) + 1)
customer = customer[["customer_id", "first_name", "last_name", "Email", "Phone"]]

In [50]:
# Create the Accounts table
account = (
    zulo_bank[["AccountType", "Balance", "OpeningDate"]]
    .copy()
    .drop_duplicates()
    .reset_index(drop=True)
)
account["account_id"] = range(1, len(account) + 1)
account = account[["account_id", "AccountType", "Balance", "OpeningDate"]]

In [51]:
# Create transactions table
transaction = (
    zulo_bank[["TransactionType", "Amount", "TransactionDate"]]
    .copy()
    .drop_duplicates()
    .reset_index(drop=True)
)
transaction["transaction_id"] = range(1, len(transaction) + 1)
transaction = transaction[
    ["transaction_id", "TransactionType", "Amount", "TransactionDate"]
]

In [52]:
# Create a Loans table
loan = (
    zulo_bank[["LoanAmount", "LoanType", "StartDate", "EndDate", "InterestRate"]]
    .copy()
    .drop_duplicates()
    .reset_index(drop=True)
)
loan["loan_id"] = range(1, len(loan) + 1)
loan = loan[
    ["loan_id", "LoanAmount", "LoanType", "StartDate", "EndDate", "InterestRate"]
]

In [53]:
# Merge operations to create the zulo_bank
zulo_bank = (
    zulo_bank.merge(
        customer, on=["first_name", "last_name", "Email", "Phone"], how="left"
    )
    .merge(account, on=["AccountType", "Balance", "OpeningDate"], how="left")
    .merge(transaction, on=["TransactionType", "Amount", "TransactionDate"], how="left")
    .merge(
        loan,
        on=["LoanAmount", "LoanType", "StartDate", "EndDate", "InterestRate"],
        how="left",
    )[["customer_id", "account_id", "transaction_id", "loan_id"]]
)

In [54]:
# zulo_bank

In [55]:
"""
Convert 2NF to 3NF: Third Normal Form
Create the date dimension table
Define the start and end dates for the date dimension
"""

start_date = datetime(2020, 1, 1)
current_date = datetime(2090, 12, 31)

# Calculate the number of days between start date and current date
num_days = (current_date - start_date).days

# Generate a list of dates from start date to current date
date_list = [start_date + timedelta(days=x) for x in range(num_days + 1)]

# Ensure date_id matches the length of date_list
date = {"date_id": [x for x in range(1, len(date_list) + 1)], "date": date_list}

# Create DataFrame
date_dim = pd.DataFrame(date)
date_dim["Year"] = date_dim["date"].dt.year
date_dim["Month"] = date_dim["date"].dt.month
date_dim["Day"] = date_dim["date"].dt.day
date_dim["date"] = pd.to_datetime(date_dim["date"]).dt.date

In [56]:
# date_dim

In [57]:
# Account table 2NF to 3NF
account["OpeningDate"] = pd.to_datetime(account["OpeningDate"]).dt.date
account = (
    account.merge(date_dim, left_on="OpeningDate", right_on="date", how="inner")
    .rename(columns={"date_id": "OpeningDate_ID"})
    .reset_index(drop=True)[["account_id", "AccountType", "Balance", "OpeningDate_ID"]]
)

In [58]:
# Create the transaction table 2NF to 3NF
transaction["TransactionDate"] = pd.to_datetime(transaction["TransactionDate"]).dt.date
transaction = (
    transaction.merge(date_dim, left_on="TransactionDate", right_on="date", how="inner")
    .rename(columns={"date_id": "TransactionDate_ID"})
    .reset_index(drop=True)[
        ["transaction_id", "TransactionType", "Amount", "TransactionDate_ID"]
    ]
)

In [59]:
# transaction

In [60]:
# Create Loan table 2NF to 3NF
loan["StartDate"] = pd.to_datetime(loan["StartDate"]).dt.date 
loan["EndDate"] = pd.to_datetime(loan["EndDate"]).dt.date
loan = (
    loan.merge(date_dim, left_on="StartDate", right_on="date", how="inner")
    .rename(columns={"date_id": "StartDate_ID"})
    .merge(
        date_dim, left_on="EndDate", right_on="date", how="inner", suffixes=("", "_end")
    )
    .rename(columns={"date_id": "EndDate_ID"})[
        [
            "loan_id",
            "LoanAmount",
            "LoanType",
            "StartDate_ID",
            "EndDate_ID",
            "InterestRate",
        ]
    ]
)

In [61]:
# loan

In [62]:
# Save the final tables to new directory
customer.to_csv(r"dataset/database_model/customers.csv", index=False)
account.to_csv(r"dataset/database_model/accounts.csv", index=False)
transaction.to_csv(r"dataset/database_model/transactions.csv", index=False)
loan.to_csv(r"dataset/database_model/loans.csv", index=False)  
date_dim.to_csv(r"dataset/database_model/date_dim.csv", index=False)
zulo_bank.to_csv(r"dataset/database_model/fact_table.csv", index=False)  
    

In [63]:
""" 
# Modeling for the data warehouse
# Transaction Data Warehouse Schema Design
"""
transaction_dim = (
    transaction[["transaction_id", "TransactionType"]]
    .copy()
    .drop_duplicates()
    .reset_index(drop=True)
)
account_dim = (
    account[["account_id", "AccountType", "Balance"]]
    .copy()
    .drop_duplicates()
    .reset_index(drop=True)
)

transaction_fact_table = zulo_bank.merge(
    transaction, on="transaction_id", how="inner"
).merge(account, on="account_id", how="inner")[
    ["transaction_id", "account_id", "OpeningDate_ID", "TransactionDate_ID", "Amount"]
]

In [64]:
# Save to memory
transaction_dim.to_csv(r"dataset/transaction_dwh/transaction_dim.csv", index=False)
account_dim.to_csv(r"dataset/transaction_dwh/account_dim.csv", index=False)
transaction_fact_table.to_csv(
    r"dataset/transaction_dwh/transaction_fact_table.csv", index=False
)

In [65]:
# transaction_dim
# account_dim
# transaction_fact_table


# Loan Database Warehouse (DWH) Schema

In [66]:
customer_dim = customer.copy()
loan_dim = loan[["loan_id", "LoanType"]].copy().drop_duplicates().reset_index(drop=True)

loan_fact_table = zulo_bank.merge(customer, on="customer_id", how="inner").merge(
    loan, on="loan_id", how="inner"
)[
    [
        "loan_id",
        "customer_id",
        "StartDate_ID",
        "EndDate_ID",
        "LoanAmount",
        "InterestRate",
    ]
]

In [67]:
# Save to memory
loan_dim.to_csv(r"dataset/loan_dwh/loan_dim.csv", index=False)
customer_dim.to_csv(r"dataset/loan_dwh/customer_dim.csv", index=False)
loan_fact_table.to_csv(r"dataset/loan_dwh/loan_fact_table.csv", index=False)

In [68]:
#bloan_dim

# Loading into the RDBMS using psycopg2

In [69]:
#%pip uninstall psycopg2 psycopg2-binary -y
#%pip install psycopg2-binary

In [70]:
import psycopg2

In [71]:
# Define database connection parameters

db_params = {
    "database": "zulo_bank",
    "username": "postgres",
    "password": "password",
    "host": "localhost",
    "port": "5432",
}


default_db_url = f"postgresql://{db_params['username']}:{db_params['password']}@{db_params['host']}:{db_params['port']}/postgres"

# Create database connection
try:
    # Open the connection
    conn = psycopg2.connect(default_db_url)
    conn.autocommit = True
    cur = conn.cursor()
    # Check if the database already exists
    cur.execute(f"SELECT 1 FROM pg_catalog.pg_database WHERE datname = '{db_params['database']}'")
    exists = cur.fetchone()
    if not exists:
        cur.execute(f"CREATE DATABASE {db_params['database']}")
        print(f"Database {db_params['database']} created successfully.")
    else:
        print(f"Database {db_params['database']} already exists.")
    # Close the connection to the default database
    cur.close()
    conn.close() 
except Exception as e:
    print(f"An error occurred: {e}") 

Database zulo_bank created successfully.


In [72]:
# Coonect to the newly created database or existing database
def get_db_connection():
    connection = psycopg2.connect(
        host="localhost", database="zulo_bank", user="postgres", password="password"
    )
    return connection


conn = get_db_connection()

In [73]:
# Create schema and tables
def create_tables():
    conn = get_db_connection()
    cursor = conn.cursor()
    create_table_query = """
                            CREATE SCHEMA IF NOT EXISTS zulobankdb;

                            DROP TABLE IF EXISTS zulobankdb.transactions CASCADE;
                            DROP TABLE IF EXISTS zulobankdb.accounts CASCADE;
                            DROP TABLE IF EXISTS zulobankdb.customers CASCADE;
                            DROP TABLE IF EXISTS zulobankdb.date_dim CASCADE;
                            DROP TABLE IF EXISTS zulobankdb.loans CASCADE;
                            DROP TABLE IF EXISTS zulobankdb.zulo_fact_table CASCADE;
                            
                            CREATE TABLE IF NOT EXISTS zulobankdb.date_dim (
                                date_id SERIAL PRIMARY KEY,
                                date VARCHAR(10000),
                                Year INTEGER,
                                Month INTEGER,
                                Day INTEGER
                            );

                            CREATE TABLE IF NOT EXISTS zulobankdb.transactions (
                                transaction_id SERIAL PRIMARY KEY,
                                TransactionType VARCHAR(10000),
                                Amount FLOAT,
                                TransactionDate_ID INTEGER,
                                FOREIGN KEY (TransactionDate_ID) REFERENCES zulobankdb.date_dim(date_id)
                            );

                            CREATE TABLE IF NOT EXISTS zulobankdb.accounts (
                                account_id SERIAL PRIMARY KEY,
                                AccountType VARCHAR(10000),
                                Balance FLOAT,
                                OpeningDate_ID INTEGER,
                                FOREIGN KEY (OpeningDate_ID) REFERENCES zulobankdb.date_dim(date_id)
                            );

                            CREATE TABLE IF NOT EXISTS zulobankdb.loans (
                                loan_id SERIAL PRIMARY KEY,
                                LoanAmount FLOAT,
                                LoanType VARCHAR(10000),
                                StartDate_ID INTEGER,
                                EndDate_ID  INTEGER,
                                InterestRate FLOAT,
                                FOREIGN KEY (StartDate_ID) REFERENCES zulobankdb.date_dim(date_id),
                                FOREIGN KEY (EndDate_ID) REFERENCES zulobankdb.date_dim(date_id)
                            );

                            CREATE TABLE IF NOT EXISTS zulobankdb.customers (
                                customer_id SERIAL PRIMARY KEY,
                                first_name VARCHAR(10000),
                                last_name VARCHAR(10000),
                                Email VARCHAR(10000),
                                Phone VARCHAR(10000)
                            );

                            CREATE TABLE IF NOT EXISTS zulobankdb.zulo_fact_table (
                                customer_id INTEGER,
                                account_id  INTEGER,
                                transaction_id INTEGER,
                                loan_id  INTEGER,
                                FOREIGN KEY (customer_id) REFERENCES zulobankdb.customers(customer_id),
                                FOREIGN KEY (account_id) REFERENCES zulobankdb.accounts(account_id),
                                FOREIGN KEY (transaction_id) REFERENCES zulobankdb.transactions(transaction_id),
                                FOREIGN KEY (loan_id) REFERENCES zulobankdb.loans(loan_id)
                            );"""
    cursor.execute(create_table_query)
    conn.commit()
    cursor.close()
    conn.close()

In [74]:
create_tables()

In [75]:
# Loading the data
# date_dim table
import csv


def load_data_from_csv(csv_path):
    conn = get_db_connection()
    cursor = conn.cursor()
    with open(csv_path, "r") as file:
        reader = csv.reader(file)
        next(reader)
        for row in reader:
            cursor.execute(
                """INSERT INTO zulobankdb.date_dim (date_id, date, Year, Month, Day)
                    VALUES (%s, %s, %s, %s, %s);""",
                row,
            )
    conn.commit()
    cursor.close()
    conn.close()


# provide  the csv path to the file
csv_file_path = r"dataset/database_model/date_dim.csv"

load_data_from_csv(csv_file_path)
print("dim_date data loaded successfully ")

dim_date data loaded successfully 


In [76]:
def load_data_from_csv(csv_path):
    conn = get_db_connection()
    cursor = conn.cursor()
    with open(csv_path, "r") as file:
        reader = csv.reader(file)
        next(reader)
        for row in reader:
            cursor.execute(
                """INSERT INTO zulobankdb.transactions (transaction_id, TransactionType, Amount, TransactionDate_ID)
                    VALUES (%s, %s, %s, %s);""",
                row,
            )
    conn.commit()
    cursor.close()
    conn.close()


# provide  the csv path to the file
csv_file_path = r"dataset/database_model/transactions.csv"

load_data_from_csv(csv_file_path)
print("transactions data loaded successfully ")

transactions data loaded successfully 


In [77]:
def load_data_from_csv(csv_path):
    conn = get_db_connection()
    cursor = conn.cursor()
    with open(csv_path, "r") as file:
        reader = csv.reader(file)
        next(reader)
        for row in reader:
            cursor.execute(
                """INSERT INTO zulobankdb.accounts (account_id, AccountType, Balance, OpeningDate_ID)
                    VALUES (%s, %s, %s, %s);""",
                row,
            )
    conn.commit()
    cursor.close()
    conn.close()


# provide  the csv path to the file
csv_file_path = r"dataset/database_model/accounts.csv"

load_data_from_csv(csv_file_path)
print("accounts data loaded successfully ")

accounts data loaded successfully 


In [78]:
def load_data_from_csv(csv_path):
    conn = get_db_connection()
    cursor = conn.cursor()
    with open(csv_path, "r") as file:
        reader = csv.reader(file)
        next(reader)
        for row in reader:
            cursor.execute(
                """INSERT INTO zulobankdb.loans (loan_id, LoanAmount, LoanType, StartDate_ID, EndDate_ID, InterestRate)
                    VALUES (%s, %s, %s, %s, %s, %s);""",
                row,
            )
    conn.commit()
    cursor.close()
    conn.close()


# provide  the csv path to the file
csv_file_path = r"dataset/database_model/loans.csv"

load_data_from_csv(csv_file_path)
print("loans data loaded successfully ")

loans data loaded successfully 


In [79]:
def load_data_from_csv(csv_path):
    conn = get_db_connection()
    cursor = conn.cursor()
    with open(csv_path, "r") as file:
        reader = csv.reader(file)
        next(reader)
        for row in reader:
            cursor.execute(
                """INSERT INTO zulobankdb.customers (customer_id, first_name, last_name, Email, Phone)
                    VALUES (%s, %s, %s, %s, %s);""",
                row,
            )
    conn.commit()
    cursor.close()
    conn.close()


# provide  the csv path to the file
csv_file_path = r"dataset/database_model/customers.csv"

load_data_from_csv(csv_file_path)
print("customers data loaded successfully ")

customers data loaded successfully 


In [80]:
def load_data_from_csv(csv_path):
    conn = get_db_connection()
    cursor = conn.cursor()
    with open(csv_path, "r") as file:
        reader = csv.reader(file)
        next(reader)
        for row in reader:
            try:
                cursor.execute(
                    """INSERT INTO zulobankdb.zulo_fact_table (customer_id, account_id, transaction_id, loan_id)
                    VALUES (%s, %s, %s, %s);""",
                    row,
                )
            except psycopg2.IntegrityError:  # Catch foregin key violation
                conn.rollback()  # Rollback the current transaction so you can continue
            else:
                conn.commit()  # Commit if the rows are inserted successfully
    cursor.close()
    conn.close()


# provide  the csv path to the file
csv_file_path = r"dataset/database_model/fact_table.csv"

load_data_from_csv(csv_file_path)
print("zulo_fact_table data loaded successfully ")

zulo_fact_table data loaded successfully 
